# CropDoc — export cotton research pilot to LiteRT/TFLite
This notebook converts the supplied `cotton_binary_pilot` MobileNetV3-Small checkpoint into a TFLite model and checks numerical parity. It does not train a new model. Predictions remain preliminary because the source labels have not been independently validated.

Upload `best_model.pt` and its matching `model_metadata.json` when prompted.


In [ ]:
!pip -q install litert-torch torchvision ai-edge-litert
print("Install completed. If importing torch/torchvision fails, restart the runtime once and rerun from the upload cell.")


In [ ]:
from google.colab import files
uploaded = files.upload()
required = {"best_model.pt", "model_metadata.json"}
missing = required - set(uploaded.keys())
if missing:
    raise RuntimeError(f"Please upload these exact files: {sorted(missing)}")
print("Uploaded checkpoint and metadata.")


In [ ]:
import hashlib, json, pathlib, numpy as np, torch, torchvision
from torch import nn
import litert_torch

CKPT = pathlib.Path("best_model.pt")
META = pathlib.Path("model_metadata.json")
OUT = pathlib.Path("cotton_binary_fp32.tflite")
MANIFEST = pathlib.Path("cotton_binary_fp32_manifest.json")
EXPECTED_SHA = "cc1fbcebaa3a80659da42905bcbdf548df4fd81815701de82ff4a9006077b33c"
CLASSES = ["cotton_bacterial_blight", "cotton_healthy"]
INPUT_SHAPE = [1, 3, 224, 224]
OUTPUT_SHAPE = [1, 2]
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

meta = json.loads(META.read_text())
actual_sha = sha256(CKPT)
assert actual_sha == EXPECTED_SHA, f"Unexpected checkpoint hash: {actual_sha}"
assert meta["checkpoint_sha256"] == actual_sha
assert meta["experiment"] == "cotton_binary_pilot"
assert meta["production_approval"] == "BLOCKED"
assert meta["diagnostic_status"] == "UNVALIDATED_PILOT (UNREVIEWED_SOURCE_LABELS)"
assert meta["classes"] == CLASSES
assert meta["input_tensor_shape"] == INPUT_SHAPE

checkpoint = torch.load(CKPT, map_location="cpu", weights_only=True)
assert checkpoint["architecture"] == "mobilenet_v3_small"
assert checkpoint["class_map"] == {CLASSES[0]: 0, CLASSES[1]: 1}
model = torchvision.models.mobilenet_v3_small(weights=None)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 2)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()
with torch.inference_mode():
    assert list(model(torch.zeros(INPUT_SHAPE)).shape) == OUTPUT_SHAPE
print("Checkpoint verified:", actual_sha)
print("Architecture: MobileNetV3-Small; output order:", CLASSES)
print("Checkpoint epoch:", checkpoint.get("epoch"), "validation samples:", checkpoint.get("val_metrics", {}).get("total_samples"))
print("Note: the checkpoint’s reported validation result is not independent confirmation of biological labels.")


In [ ]:
def as_numpy(value):
    if isinstance(value, (tuple, list)):
        if len(value) != 1:
            raise RuntimeError(f"Expected one output, got {len(value)}")
        value = value[0]
    if isinstance(value, torch.Tensor):
        value = value.detach().cpu().numpy()
    return np.asarray(value)

torch.manual_seed(1234)
sample = (torch.zeros(INPUT_SHAPE, dtype=torch.float32),)
edge_model = litert_torch.convert(model, sample)
rng = np.random.default_rng(1234)
max_abs_err = 0.0
top1_matches = 0
cases = 8
with torch.inference_mode():
    for _ in range(cases):
        x = torch.from_numpy((rng.standard_normal(INPUT_SHAPE) * 0.75).astype(np.float32))
        py = model(x).detach().cpu().numpy()
        rt = as_numpy(edge_model(x))
        if list(rt.shape) != OUTPUT_SHAPE:
            raise RuntimeError(f"Converted output shape {list(rt.shape)} != {OUTPUT_SHAPE}")
        max_abs_err = max(max_abs_err, float(np.max(np.abs(py - rt))))
        top1_matches += int(np.argmax(py, axis=1)[0] == np.argmax(rt, axis=1)[0])
if max_abs_err > 0.05:
    raise RuntimeError(f"Parity check failed: max absolute logit error {max_abs_err:.6f} > 0.05")
edge_model.export(str(OUT))
header = OUT.read_bytes()[:8]
if len(header) < 8 or header[4:8] != b"TFL3":
    OUT.unlink(missing_ok=True)
    raise RuntimeError("Exported artifact does not have TFL3 FlatBuffer magic")

manifest = {
    "model_id": "cotton_binary_fp32",
    "model_version": "0.1.0-research",
    "research_only": True,
    "production_approval": "BLOCKED",
    "diagnostic_status": "UNVALIDATED_PILOT (UNREVIEWED_SOURCE_LABELS)",
    "checkpoint_sha256": actual_sha,
    "sha256": sha256(OUT),
    "input_shape": INPUT_SHAPE, "input_dtype": "float32",
    "output_shape": OUTPUT_SHAPE, "output_dtype": "float32",
    "input_layout": "NCHW", "class_order": CLASSES,
    "normalization": {"mean": MEAN, "std": STD},
    "conversion_check": {"max_abs_logit_error": max_abs_err, "tolerance": 0.05, "synthetic_parity_cases": cases, "synthetic_top1_matches": top1_matches, "synthetic_top1_match_is_not_accuracy": True},
    "warning": "Research-only cotton pilot. Source labels are not pathologist-certified. Not approved for production diagnosis or release."
}
MANIFEST.write_text(json.dumps(manifest, indent=2) + "\n")
print("Export complete:", OUT, OUT.stat().st_size, "bytes")
print("SHA-256:", manifest["sha256"])
print("Max abs logit parity error:", max_abs_err)
print("Synthetic top-1 matches:", top1_matches, "/", cases, "(NOT model accuracy)")
print("Manifest:", MANIFEST)
print("PRODUCTION APPROVAL REMAINS BLOCKED.")


In [ ]:
from google.colab import files
files.download(str(OUT))
files.download(str(MANIFEST))


## Install the exported model in CropDoc
Copy both downloaded files into `app/src/main/assets/research_models/` using the included `tools/copy_cotton_model_to_main_assets.ps1` script. The model remains explicitly marked `research_only` and `production_approval: BLOCKED`; this enables the crop-selected Scan model bundle to run the current Cotton pilot output, not a verified production diagnosis.
